# 07 — Efficient Attention for Long Sequences
### Starter Notebook

**Learning objectives**
- Understand the $O(n^2)$ bottleneck of standard attention
- Implement sliding-window (local) attention
- Understand the FlashAttention idea (IO-aware computation)
- Benchmark memory and speed improvements

---

In [ ]:
import sys, os, math, time
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Part A — The $O(n^2)$ Problem

Standard attention computes an $n \times n$ matrix (query × key) — memory and compute both scale **quadratically** with sequence length.

| Seq length | Attention matrix | Memory (fp16) |
|------------|-----------------|---------------|
| 512        | 512×512        | 0.5 MB        |
| 2048       | 2048×2048      | 8 MB          |
| 8192       | 8192×8192      | 128 MB        |
| 32768      | 32768×32768    | 2 GB          |

### Exercise A1 — Measure the quadratic scaling

In [ ]:
from src.models.attention import MultiHeadAttention

def measure_attention_memory(seq_len: int, d_model: int = 128, n_heads: int = 4) -> float:
    """Return peak GPU memory (MB) for a single attention forward pass."""
    if not torch.cuda.is_available():
        # CPU: estimate from tensor size
        # Attention matrix: (batch=1, heads, seq, seq) * 4 bytes (fp32)
        return n_heads * seq_len * seq_len * 4 / 1e6

    torch.cuda.reset_peak_memory_stats()
    attn = MultiHeadAttention(d_model, n_heads).to('cuda')
    x = torch.randn(1, seq_len, d_model, device='cuda')
    with torch.no_grad():
        attn(x, x, x)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 1e6


seq_lengths = [64, 128, 256, 512, 1024]
memories = [measure_attention_memory(s) for s in seq_lengths]

plt.figure(figsize=(8, 4))
plt.plot(seq_lengths, memories, 'o-', color='tomato')
# TODO: add a reference O(n^2) line to confirm the scaling
plt.xlabel('Sequence length'); plt.ylabel('Memory (MB)')
plt.title('Attention Memory Scaling (should be ~quadratic)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Extrapolate to longer sequences
for s in [2048, 8192, 32768]:
    # TODO: extrapolate memory assuming O(n^2) scaling
    pass

## Part B — Sliding Window Attention

Each token attends only to a fixed-size local window around it. Complexity drops from $O(n^2)$ to $O(n \cdot w)$ where $w$ is the window size.

Used in **Mistral** (window size = 4096).

### Exercise B1 — Build the sliding window mask

In [ ]:
def make_sliding_window_mask(seq_len: int, window_size: int, causal: bool = True) -> torch.Tensor:
    """
    Create a sliding window attention mask.

    Args:
        seq_len    : sequence length
        window_size: positions on each side that can be attended to
        causal     : if True, also apply causal (lower-triangular) mask

    Returns:
        mask: (1, 1, seq_len, seq_len)  — 1=attend, 0=block
    """
    # TODO:
    # 1. Create position distance matrix |i - j|
    # 2. Mask positions where distance > window_size
    # 3. Optionally apply causal mask (j <= i)
    pass


mask = make_sliding_window_mask(seq_len=12, window_size=3, causal=True)
if mask is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, causal, title in zip(axes, [False, True], ['Bidirectional (window=3)', 'Causal (window=3)']):
        m = make_sliding_window_mask(12, 3, causal=causal)
        ax.imshow(m.squeeze(), cmap='Blues', vmin=0, vmax=1)
        ax.set_title(title); ax.set_xlabel('Key pos'); ax.set_ylabel('Query pos')
    plt.tight_layout(); plt.show()
else:
    print('Not implemented yet.')

### Exercise B2 — Use the library implementation

In [ ]:
from src.models.efficient_attention import SlidingWindowAttention

swa = SlidingWindowAttention(d_model=128, n_heads=4, window_size=32)
x = torch.randn(2, 128, 128)   # batch=2, seq=128
out, _ = swa(x)
print(f'SlidingWindowAttention output: {out.shape}')

# Compare memory vs standard attention for the same sequence
# TODO: run both and compare peak memory or estimated attention matrix size
seq = 256
full_attn_mat = seq * seq
window = 32
sliding_attn_mat = seq * (2 * window + 1)
print(f'\nAttention matrix elements (seq={seq}, window={window}):')
print(f'  Full attention  : {full_attn_mat:,}')
print(f'  Sliding window  : {sliding_attn_mat:,}   ({full_attn_mat/sliding_attn_mat:.1f}x smaller)')

## Part C — FlashAttention Concept

FlashAttention (Dao et al., 2022) doesn't change the *mathematical* result of attention — it changes *how it is computed* to be memory-efficient.

**Key insight:** GPUs have a fast on-chip SRAM (shared memory) and a slow off-chip HBM (main GPU memory). Standard attention writes the full $n \times n$ matrix to HBM. FlashAttention computes attention in **tiles** that fit in SRAM, never materialising the full matrix.

**Result:** Memory scales $O(n)$ instead of $O(n^2)$.

In [ ]:
# PyTorch 2.0+ includes a FlashAttention-compatible implementation
print(f'PyTorch version: {torch.__version__}')
print(f'SDPA (scaled_dot_product_attention) available: {hasattr(F, "scaled_dot_product_attention")}')

if hasattr(F, 'scaled_dot_product_attention'):
    # This uses FlashAttention automatically when on CUDA and inputs allow it
    B, H, S, dk = 2, 4, 64, 32
    q = torch.randn(B, H, S, dk, device=device)
    k = torch.randn(B, H, S, dk, device=device)
    v = torch.randn(B, H, S, dk, device=device)

    with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=True, enable_mem_efficient=True) if device.type == 'cuda' else torch.no_grad():
        out_flash = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    print(f'PyTorch SDPA output: {out_flash.shape}')

    # Verify it matches our manual implementation
    from src.models.attention import scaled_dot_product_attention, create_causal_mask
    mask = create_causal_mask(S, device=device)
    out_manual, _ = scaled_dot_product_attention(q, k, v, mask=mask)
    print(f'Results match: {torch.allclose(out_flash, out_manual, atol=1e-4)}')

## Part D — Linear Attention (Approximation)

Another approach: **approximate** the softmax with a kernel trick to achieve $O(n)$ complexity.

The idea: $\text{softmax}(QK^T)V \approx \phi(Q)(\phi(K)^T V)$ where $\phi$ is a feature map.

By computing the right-to-left associativity, we get $O(n \cdot d^2)$ instead of $O(n^2 \cdot d)$.

In [ ]:
from src.models.efficient_attention import LinearAttention

linear_attn = LinearAttention(d_model=128, n_heads=4)
x = torch.randn(2, 256, 128)
out, _ = linear_attn(x)
print(f'LinearAttention output: {out.shape}')

# Complexity comparison
print('\nComplexity comparison:')
print('  Standard attention : O(n² · d)  — quadratic in sequence length')
print('  Sliding window     : O(n · w · d) — linear, but limited receptive field')
print('  Linear attention   : O(n · d²)  — linear, full context but approximate')
print('  FlashAttention     : O(n²)      — same math, much less memory I/O')

## Summary

| Method | Complexity | Exact? | Used in |
|--------|-----------|--------|--------|
| Standard | $O(n^2 d)$ | Yes | All transformers |
| Sliding Window | $O(nwd)$ | Yes (local) | Mistral, Longformer |
| FlashAttention | $O(n^2)$ (memory $O(n)$) | Yes | LLaMA, Gemma |
| Linear | $O(nd^2)$ | Approx | Research models |

**Next:** `08_mixture_of_experts_starter.ipynb` — scale model capacity without scaling compute.